# Beginner Hands-on Lab: DBSCAN Clustering
**Goal:** Find clusters and identify core, border, and noise points. No dataset upload is needed; this notebook creates a small practice dataset.

You will learn:
- `eps`: how far to look for neighbors.
- `min_samples`: the minimum neighborhood count for a core point, including itself.
- Core points expand clusters, border points can join clusters, and noise points remain unassigned.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN

print("Ready!")

## 2. Create a simple dataset
We create two curved groups plus a few random outliers, so DBSCAN has clusters and possible noise to find.

In [ ]:
X, _ = make_moons(n_samples=250, noise=0.07, random_state=42)
rng = np.random.RandomState(42)
outliers = rng.uniform(low=-1.5, high=2.5, size=(12, 2))
X = np.vstack([X, outliers])
data = pd.DataFrame(X, columns=["Feature 1", "Feature 2"])
print("Number of points:", len(data))
display(data.head())

## 3. Plot before clustering

In [ ]:
plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=30)
plt.title("Before DBSCAN: unlabeled points")
plt.xlabel("Feature 1"); plt.ylabel("Feature 2")
plt.grid(alpha=0.25); plt.show()

## 4. Scale features
DBSCAN uses distances. Scaling is especially important when features have different units or ranges.

In [ ]:
X_scaled = StandardScaler().fit_transform(X)
print("Scaled data shape:", X_scaled.shape)

## 5. Run DBSCAN
Start with `eps = 0.25` and `min_samples = 5`. These are starting values for this practice dataset, not universal best values.

In [ ]:
eps_value = 0.25
min_samples_value = 5

model = DBSCAN(eps=eps_value, min_samples=min_samples_value)
labels = model.fit_predict(X_scaled)
data["Cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
n_noise = int(np.sum(labels == -1))
print("eps:", eps_value, "| min_samples:", min_samples_value)
print("Clusters found:", n_clusters)
print("Noise points:", n_noise)
display(data["Cluster"].value_counts().sort_index().to_frame("Point count"))

## 6. Visualize clusters
Each color is a cluster. Black crosses (`-1`) are noise points.

In [ ]:
plt.figure(figsize=(8, 5))
for label in sorted(set(labels)):
    mask = labels == label
    if label == -1:
        plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1], c="black",
                    marker="x", s=65, label="Noise (-1)")
    else:
        plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1], s=32,
                    label=f"Cluster {label}")
plt.title(f"DBSCAN result: eps={eps_value}, min_samples={min_samples_value}")
plt.xlabel("Scaled Feature 1"); plt.ylabel("Scaled Feature 2")
plt.legend(); plt.grid(alpha=0.2); plt.show()

## 7. Identify core, border, and noise points
Scikit-learn gives us the indices of core points. Border points are assigned to a cluster but are not core points. Noise points have label `-1`.

In [ ]:
is_core = np.zeros(len(X_scaled), dtype=bool)
is_core[model.core_sample_indices_] = True
is_noise = labels == -1
is_border = (~is_core) & (~is_noise)

print("Core points:", int(is_core.sum()))
print("Border points:", int(is_border.sum()))
print("Noise points:", int(is_noise.sum()))

plt.figure(figsize=(8, 5))
plt.scatter(X_scaled[is_border, 0], X_scaled[is_border, 1],
            c="orange", s=42, label="Border points")
plt.scatter(X_scaled[is_core, 0], X_scaled[is_core, 1],
            c="royalblue", s=35, label="Core points")
plt.scatter(X_scaled[is_noise, 0], X_scaled[is_noise, 1],
            c="black", marker="x", s=65, label="Noise points")
plt.title("DBSCAN point types")
plt.xlabel("Scaled Feature 1"); plt.ylabel("Scaled Feature 2")
plt.legend(); plt.grid(alpha=0.2); plt.show()

## 8. Experiment with `eps` and `min_samples`
Change the values in the next cell and rerun it. Try `eps_try = 0.15`, `0.30`, and `0.40`; then compare `min_samples_try = 3`, `5`, and `10`.

In [ ]:
eps_try = 0.30
min_samples_try = 5

trial_model = DBSCAN(eps=eps_try, min_samples=min_samples_try)
trial_labels = trial_model.fit_predict(X_scaled)
trial_clusters = len(set(trial_labels)) - (1 if -1 in trial_labels else 0)
trial_noise = int(np.sum(trial_labels == -1))

print(f"eps={eps_try}, min_samples={min_samples_try}")
print("Clusters:", trial_clusters)
print("Noise points:", trial_noise)

plt.figure(figsize=(8, 5))
for label in sorted(set(trial_labels)):
    mask = trial_labels == label
    if label == -1:
        plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1], c="black",
                    marker="x", s=65, label="Noise (-1)")
    else:
        plt.scatter(X_scaled[mask, 0], X_scaled[mask, 1], s=32,
                    label=f"Cluster {label}")
plt.title(f"DBSCAN: eps={eps_try}, min_samples={min_samples_try}")
plt.xlabel("Scaled Feature 1"); plt.ylabel("Scaled Feature 2")
plt.legend(); plt.grid(alpha=0.2); plt.show()

## 9. Student practice tasks
1. Set `eps_try = 0.15`, `min_samples_try = 5`. What changes?
2. Set `eps_try = 0.40`, `min_samples_try = 5`. Do groups connect?
3. Keep `eps_try = 0.30`; compare `min_samples_try = 3` and `10`.
4. What does the black cross (`-1`) mean?
5. Explain core point vs border point in one sentence.

### Quick takeaway
- **Small `eps`:** neighborhoods are smaller; clusters may split or more points may become noise.
- **Large `eps`:** neighborhoods are larger; clusters may connect or merge.
- **Higher `min_samples`:** a denser neighborhood is needed to qualify as a core point.
- Results depend on the data; these are common tendencies, not guarantees.